In [1]:
import pandas as pd

df = pd.read_csv("data/train_test.csv", parse_dates=["date"])
print(df.shape)
df.head()

(48000, 14)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [2]:
df.isna().sum()  
df.describe()            # min, max, average of the number columns
df["date"].min(), df["date"].max()   # date range
df["equipment"].value_counts()       # how many of each equipment type

equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64

In [3]:
train = df[df["date"] < "2025-09-01"]
test  = df[df["date"] >= "2025-09-01"]

print("train:", train.shape, train["date"].min(), "→", train["date"].max())
print("test: ", test.shape,  test["date"].min(),  "→", test["date"].max())

train: (38477, 14) 2025-01-01 00:00:00 → 2025-08-31 00:00:00
test:  (9523, 14) 2025-09-01 00:00:00 → 2025-10-31 00:00:00


In [4]:
import numpy as np

# Evaluation helper: we'll reuse this for every model
def evaluate(y_true, y_pred, name=""):
    mae  = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs(y_true - y_pred) / y_true) * 100
    print(f"{name:35s} MAE: ${mae:,.0f}   MAPE: {mape:.1f}%")

# Learn the typical rate per mile from TRAIN only
train = train.copy()
train["rpm"] = train["posted_rate"] / train["distance"]
median_rpm = train["rpm"].median()      # median, so extreme rates don't distort it
print("median rate per mile:", round(median_rpm, 3))

# Predict on TEST
pred = test["distance"] * median_rpm
evaluate(test["posted_rate"], pred, "Baseline 1: distance x median rpm")

median rate per mile: 2.141
Baseline 1: distance x median rpm   MAE: $257   MAPE: 11.6%


In [5]:
rpm_by_equip = train.groupby("equipment")["rpm"].median()
print(rpm_by_equip)

pred2 = test["distance"] * test["equipment"].map(rpm_by_equip)
evaluate(test["posted_rate"], pred2, "Baseline 2: + equipment")

equipment
Dry Van    2.044714
Flatbed    2.213186
Reefer     2.310283
Name: rpm, dtype: float64
Baseline 2: + equipment             MAE: $229   MAPE: 10.5%


In [6]:
rpm_all = df["posted_rate"] / df["distance"]

print("negative weights:", (df["weight"] < 0).sum())
print("abs(weight) range:", df["weight"].abs().min(), df["weight"].abs().max())
print()
print("rate-per-mile percentiles:")
print(rpm_all.quantile([0.001, 0.01, 0.5, 0.99, 0.999]))
print("rows with rpm > 5:", (rpm_all > 5).sum())
print("rows with rpm < 1:", (rpm_all < 1).sum())

negative weights: 292
abs(weight) range: 5000.0 47500.0

rate-per-mile percentiles:
0.001     0.437364
0.010     1.673521
0.500     2.145348
0.990     3.176940
0.999    10.068775
dtype: float64
rows with rpm > 5: 317
rows with rpm < 1: 329


In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor

CATS = ["equipment", "pickup", "delivery"]
cat_levels = {c: sorted(df[c].unique()) for c in CATS}

def make_features(d):
    X = pd.DataFrame(index=d.index)
    X["distance"]     = d["distance"]
    X["weight"]       = d["weight"].abs()          # fix negative weights
    X["market_index"] = d["market_index"]
    X["quote_signal"] = d["quote_signal"]
    for c in ["pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]:
        X[c] = d[c]
    X["dow"] = d["date"].dt.dayofweek
    for c in CATS:
        X[c] = pd.Categorical(d[c], categories=cat_levels[c])
    return X

# Train: drop extreme rate-per-mile rows (TRAIN ONLY, never the test set)
tr = train.copy()
tr["rpm"] = tr["posted_rate"] / tr["distance"]
tr = tr[(tr["rpm"] > 0.8) & (tr["rpm"] < 6)]

model = HistGradientBoostingRegressor(
    categorical_features="from_dtype",
    learning_rate=0.05, max_iter=400, random_state=42,
)
model.fit(make_features(tr), np.log(tr["rpm"]))

pred = np.exp(model.predict(make_features(test))) * test["distance"]
evaluate(test["posted_rate"], pred, "Gradient boosting v1")

Gradient boosting v1                MAE: $126   MAPE: 5.4%


In [8]:
def make_features(d, use_city_names=True):
    X = pd.DataFrame(index=d.index)
    X["distance"]     = d["distance"]
    X["weight"]       = d["weight"].abs()
    X["market_index"] = d["market_index"]
    X["quote_signal"] = d["quote_signal"]
    for c in ["pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]:
        X[c] = d[c]
    X["dow"] = d["date"].dt.dayofweek
    X["equipment"] = pd.Categorical(d["equipment"], categories=cat_levels["equipment"])
    if use_city_names:
        for c in ["pickup", "delivery"]:
            X[c] = pd.Categorical(d[c], categories=cat_levels[c])
    return X

for flag in [True, False]:
    m = HistGradientBoostingRegressor(
        categorical_features="from_dtype",
        learning_rate=0.05, max_iter=400, random_state=42,
    )
    m.fit(make_features(tr, flag), np.log(tr["rpm"]))
    p = np.exp(m.predict(make_features(test, flag))) * test["distance"]
    evaluate(test["posted_rate"], p, f"city names = {flag}")
    

city names = True                   MAE: $126   MAPE: 5.4%
city names = False                  MAE: $130   MAPE: 5.6%


In [9]:
full = df.copy()
full["rpm"] = full["posted_rate"] / full["distance"]
full = full[(full["rpm"] > 0.8) & (full["rpm"] < 6)]

final_model = HistGradientBoostingRegressor(
    categorical_features="from_dtype",
    learning_rate=0.05, max_iter=400, random_state=42,
)
final_model.fit(make_features(full, use_city_names=False), np.log(full["rpm"]))
print("trained on", len(full), "rows")

trained on 47464 rows


In [10]:
val  = pd.read_csv("data/validation.csv", parse_dates=["date"])
tmpl = pd.read_csv("data/validation_predictions_template.csv")

val["predicted_rate"] = np.exp(
    final_model.predict(make_features(val, use_city_names=False))
) * val["distance"]

out = tmpl[["load_id"]].merge(val[["load_id", "predicted_rate"]], on="load_id", how="left")
assert len(out) == 12000 and out["predicted_rate"].notna().all() and (out["predicted_rate"] > 0).all()
out.to_csv("validation_predictions.csv", index=False)
print(out.head()); print(out["predicted_rate"].describe())

     load_id  predicted_rate
0  TE-000001      817.386049
1  TE-000002     4887.197260
2  TE-000003     5324.698271
3  TE-000004     4175.790577
4  TE-000005     1787.921925
count    12000.000000
mean      2323.630970
std       1358.495837
min        177.735726
25%       1245.398232
50%       1990.995232
75%       3327.157350
max       6688.827858
Name: predicted_rate, dtype: float64


In [11]:
dec = pd.read_csv("data/december_chart_inputs.csv")
dec_dates = pd.to_datetime(dec["date"])

daily = val.groupby("date")[["market_index", "quote_signal"]].mean()

lex = df[df["pickup"] == "Lexington"][["pickup_lat", "pickup_lon"]].median()
fw  = df[df["delivery"] == "Fort Wayne"][["delivery_lat", "delivery_lon"]].median()

X_dec = pd.DataFrame({
    "date": dec_dates,
    "distance": dec["distance"].astype(float),
    "weight": dec["weight"].astype(float),
    "equipment": dec["equipment"],
    "pickup_lat": lex["pickup_lat"], "pickup_lon": lex["pickup_lon"],
    "delivery_lat": fw["delivery_lat"], "delivery_lon": fw["delivery_lon"],
    "market_index": dec_dates.map(daily["market_index"]).values,
    "quote_signal": dec_dates.map(daily["quote_signal"]).values,
})

dec["predicted_rate"] = np.exp(
    final_model.predict(make_features(X_dec, use_city_names=False))
) * X_dec["distance"]
dec.to_csv("data/december_chart_inputs.csv", index=False)
print(dec[["date", "predicted_rate"]].round(1).to_string())

          date  predicted_rate
0   2025-12-01           800.8
1   2025-12-02           803.5
2   2025-12-03           810.5
3   2025-12-04           812.0
4   2025-12-05           810.5
5   2025-12-06           802.4
6   2025-12-07           802.9
7   2025-12-08           800.8
8   2025-12-09           802.8
9   2025-12-10           810.5
10  2025-12-11           812.0
11  2025-12-12           810.5
12  2025-12-13           801.3
13  2025-12-14           802.9
14  2025-12-15           801.4
15  2025-12-16           805.1
16  2025-12-17           810.5
17  2025-12-18           812.0
18  2025-12-19           810.5
19  2025-12-20           801.3
20  2025-12-21           803.6
21  2025-12-22           801.8
22  2025-12-23           805.4
23  2025-12-24           811.8
24  2025-12-25           812.0
25  2025-12-26           810.5
26  2025-12-27           802.7
27  2025-12-28           803.6
28  2025-12-29           801.0
29  2025-12-30           806.3
30  2025-12-31           812.0


In [12]:
lane = df[(df["pickup"]=="Lexington") & (df["delivery"]=="Fort Wayne")]
print("Lexington -> Fort Wayne in training:", len(lane), "loads")
print(lane.groupby("equipment")["posted_rate"].median().round(0))
print()
d = df[(df["posted_rate"]/df["distance"]).between(0.8, 6)].copy()
d["rpm"] = d["posted_rate"]/d["distance"]
print(d.groupby(d["date"].dt.day_name())["rpm"].mean().round(3))

Lexington -> Fort Wayne in training: 32 loads
equipment
Dry Van    808.0
Flatbed    878.0
Reefer     934.0
Name: posted_rate, dtype: float64

date
Friday       2.201
Monday       2.172
Saturday     2.178
Sunday       2.169
Thursday     2.213
Tuesday      2.188
Wednesday    2.202
Name: rpm, dtype: float64
